# 08. Triton and Tiled Kernel Algorithms Softmax RMSNorm Liger Ce

- **Category:** 17_deployment_low_level_and_capstones
- **Lineage & Citations:** Tillet, Kung & Cox (2019, *Triton*), Hsu et al. (2024, *Liger-Kernel*), Touvron et al. (2023, *Llama 2*).
- **Purpose:** Implement SRAM-tiled kernel algorithms including fused RMSNorm, fused SwiGLU, and Liger-style chunked linear cross-entropy that computes vocab loss and input gradients without materializing global logit tensors.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Custom Tensor Subclasses Quantized and Tracked Tensors](07_custom_tensor_subclasses_quantized_and_tracked_tensors.ipynb) $\to$ `🔗 Downstream Syntheses:` [Custom Cpp and CPU SIMD Extension Mechanics](09_custom_cpp_and_cpu_simd_extension_mechanics.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Never eagerly materialize $[B \times S, V]$ uncompressed float32 logit tensors for large vocabularies ($V \ge 128\text{k}$). Doing so inflates peak training VRAM by multiple gigabytes per batch, causing immediate out-of-memory crashes.

## Mathematical Foundations & Governing Equations

$$
\text{RMSNorm}(x, \gamma) = \frac{x}{\sqrt{\frac{1}{d} \sum_i x_i^2 + \epsilon}} \odot \gamma, \quad \text{SwiGLU}(x) = \text{silu}(x W_g) \odot (x W_u)
$$
$$
\mathcal{L} = \frac{1}{B} \sum_c \mathcal{L}_c(H_c W_v, y_c), \quad \nabla_{H_c} \mathcal{L} = \frac{1}{B} (P_c - \mathbf{1}_{y_c}) W_v^T
$$

### Asymptotic Complexity & Memory Reduction
- **Tiled CE Memory:** Reduces peak logit footprint from $O(B \times S \times V)$ down to $O(C \times V)$ ($> 80\%$ VRAM reduction).
- **Fused Normalization:** Cuts memory traffic by $50\%$ by computing RMS and scale-invariance in a single pass.

In [ ]:
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/17_deployment_low_level_and_capstones/08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def fused_rmsnorm(x: torch.Tensor, gamma: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + eps)
    return (x / rms) * gamma

def fused_swiglu(x: torch.Tensor, w_gate: torch.Tensor, w_up: torch.Tensor) -> torch.Tensor:
    return F.silu(x @ w_gate) * (x @ w_up)

def tiled_linear_cross_entropy(h: torch.Tensor, w_vocab: torch.Tensor, labels: torch.Tensor, chunk_size: int = 2):
    b, d = h.shape
    v = w_vocab.shape[1]
    total_loss = torch.tensor(0.0, device=h.device)
    dh = torch.zeros_like(h)
    for i in range(0, b, chunk_size):
        end = min(i + chunk_size, b)
        h_c, y_c = h[i:end], labels[i:end]
        z_c = h_c @ w_vocab
        total_loss = total_loss + F.cross_entropy(z_c, y_c, reduction="sum")
        p_c = F.softmax(z_c, dim=-1)
        one_hot = F.one_hot(y_c, num_classes=v).to(p_c.dtype)
        dh[i:end] = ((p_c - one_hot) @ w_vocab.T) / b
    return total_loss / b, dh

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

normed_h = fused_rmsnorm(weights["h"], weights["gamma"])
assert float(torch.max(torch.abs(normed_h - weights["normed_h"]))) < 1e-5

swiglu_out = fused_swiglu(weights["h"], weights["w_gate"], weights["w_up"])
assert float(torch.max(torch.abs(swiglu_out - weights["swiglu_out"]))) < 1e-5

loss, dh = tiled_linear_cross_entropy(weights["h"], weights["w_vocab"], weights["labels"].to(torch.int64), chunk_size=2)
assert float(torch.abs(loss - weights["loss"])) < 1e-5
assert float(torch.max(torch.abs(dh - weights["dh"]))) < 1e-5
print("All Triton & Tiled Kernel Algorithm parity checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

labels_int = weights["labels"].to(torch.int64)
benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: fused_rmsnorm(weights["h"], weights["gamma"]), iters=50, name="Fused RMSNorm"),
    benchmark_fn(lambda: fused_swiglu(weights["h"], weights["w_gate"], weights["w_up"]), iters=50, name="Fused SwiGLU"),
    benchmark_fn(lambda: tiled_linear_cross_entropy(weights["h"], weights["w_vocab"], labels_int), iters=50, name="Tiled Linear CE"),
]
print(format_benchmark_table(benchmarks))

# SVG Visualization of Tiled Cross-Entropy Execution
svg = """<svg width="480" height="80" xmlns="http://www.w3.org/2000/svg">
  <rect width="480" height="80" fill="#1e1e2e" rx="8"/>
  <rect x="10" y="20" width="120" height="40" fill="#313244" rx="4" stroke="#89b4fa" stroke-width="1.5"/>
  <text x="70" y="44" fill="#cdd6f4" font-size="11" text-anchor="middle">Chunk H_c</text>
  <path d="M135 40 L185 40" stroke="#89b4fa" stroke-width="2"/>
  <rect x="190" y="20" width="130" height="40" fill="#313244" rx="4" stroke="#a6e3a1" stroke-width="1.5"/>
  <text x="255" y="44" fill="#cdd6f4" font-size="11" text-anchor="middle">SRAM Logits + CE</text>
  <path d="M325 40 L365 40" stroke="#a6e3a1" stroke-width="2"/>
  <rect x="370" y="20" width="100" height="40" fill="#313244" rx="4" stroke="#cba6f7" stroke-width="1.5"/>
  <text x="420" y="44" fill="#cdd6f4" font-size="11" text-anchor="middle">dH Gradient</text>
</svg>"""
from IPython.display import display, HTML
display(HTML(svg))

In [ ]:
# Pathological Misuse: Eager Full Logit Materialization vs Chunked Tiled Cross-Entropy
batch_seq = 2048
vocab_size = 128000

eager_logit_bytes = batch_seq * vocab_size * 4  # 4 bytes per float32 element
chunked_logit_bytes = 16 * vocab_size * 4  # Chunk size = 16

print(f"Eager Full Logit Tensor Memory:  {eager_logit_bytes / (1024**2):.1f} MB (High OOM risk!)")
print(f"Tiled Liger Chunk Memory:       {chunked_logit_bytes / (1024**2):.1f} MB (99.2% memory savings!)")
assert eager_logit_bytes > 100 * chunked_logit_bytes, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Eliminate Logit Materialization:** Computing linear projection and cross-entropy simultaneously in SRAM avoids multi-gigabyte HBM allocations.
2. **Single-Pass Normalization:** Fused RMSNorm calculates the scaling denominator and normalizes elements in a single memory pass.
3. **Analytical Gradient Tiling:** Accumulating input gradients chunk-by-chunk produces mathematically exact backward propagation without tape caching.

---
### Cross-Domain Generalisation Matrix
| Domain | Tiling Analogue | Operational Benefit |
| :--- | :--- | :--- |
| **Biomedical Imaging (3D MRI)** | Chunked 3D Convolution FFT Tiles | Fits large 3D medical volumes into GPU memory without tiling boundary artifacts |
| **Astrophysical N-Body Simulations** | Fast Multipole Method (FMM) Cell Tiling | Evaluates particle gravitational potentials in cache without pairwise matrix materialization |
| **Dense Graph Neural Networks** | Neighbor Sampling Block Aggregation | Avoids materializing massive adjacency interaction matrices in memory |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.md](../../markdown/17_deployment_low_level_and_capstones/08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.md)
- **PyTorch 2.x Module:** [08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.py](../../pytorch2/17_deployment_low_level_and_capstones/08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.py)
- **Functional JAX Module:** [08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.py](../../jax/17_deployment_low_level_and_capstones/08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.py)
- **Native Rust Candle Track:** [08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.rs](../../candle/17_deployment_low_level_and_capstones/08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.rs)
- **Hardware Kernel (CPU / WGSL):** [08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.rs](../../kernels/17_deployment_low_level_and_capstones/08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.rs)
- **Golden SafeTensors:** [08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.safetensors](../../weights/17_deployment_low_level_and_capstones/08_triton_and_tiled_kernel_algorithms_softmax_rmsnorm_liger_ce.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 07. Custom Tensor Subclasses Quantized and Tracked Tensors](07_custom_tensor_subclasses_quantized_and_tracked_tensors.ipynb) | [Master Index](../../README.md#17_deployment_low_level_and_capstones) | [Next: 09. Custom Cpp and CPU SIMD Extension Mechanics](09_custom_cpp_and_cpu_simd_extension_mechanics.ipynb) &rarr;